![image.png](https://i.imgur.com/a3uAqnb.png)

In [ ]:
# Cell 1: Install required packages
!pip install ultralytics roboflow opencv-python --quiet

In [ ]:
# Cell 2: Import libraries
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import os
import random
import numpy as np
from roboflow import Roboflow
from IPython.display import Image, display
from google.colab.patches import cv2_imshow # Colab-specific display function

In [ ]:
# Cell 3: Download Segmentation Dataset from Roboflow Universe

rf = Roboflow(api_key="2dJeWPtOuxQmN2hTMrIr")

project = rf.workspace("roboflow-universe-projects").project("fire-and-smoke-segmentation")

dataset = project.version(6).download("yolov8")

In [ ]:
# Path to the dataset configuration file (data.yaml)
DATA_PATH = os.path.join(dataset.location, "data.yaml")
print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# Cell 4: Define Class Names and Inspect Data

# These are the class names for the 'Fire and Smoke Segmentation' project
class_names = ['fire', 'smoke']

print(f"Class Names: {class_names}")

train_image_dir = os.path.join(dataset.location, 'train', 'images')
random_image = random.choice(os.listdir(train_image_dir))
display(Image(filename=os.path.join(train_image_dir, random_image), width=400))

In [ ]:
!find . -name "*.cache" -delete
print("All *.cache files have been deleted from the current directory and its subfolders.")

In [ ]:
# Cell 5: Train the YOLOv8 Instance Segmentation Model

# Load the YOLO segmentation model (yolov8n-seg.pt)

# --- Ensure all cache files are deleted before training ---
# This is crucial to prevent EOFError due to corrupted cache files.
# Using `!find` is more comprehensive than Python's glob for this task across subdirectories.
!find . -name "*.cache" -delete
print("All *.cache files have been deleted from the current directory and its subfolders.")

model = YOLO('yolov8n-seg.pt')

!yolo task=segment \
    mode=train \
    model=yolov8n-seg.pt \
    data={DATA_PATH} \
    epochs=5 \
    imgsz=640 \
    batch=16 \
    name=fire_and_smoke \
    cache=False  # Explicitly disable caching for this run

# Define the path to the best model weights after training
BEST_MODEL_PATH = f'{os.getcwd()}/runs/segment/fire_and_smoke/weights/best.pt'
print(f"Trained model weights path: {BEST_MODEL_PATH}")

### Abstracted Code (Using YOLO's Built-in Plotting)

This cell demonstrates running inference and displaying the results using YOLO's integrated `plot()` method, which abstracts away the manual drawing of masks and bounding boxes. This is generally the more straightforward approach for visualization unless specific custom rendering is required.

In [ ]:
# Load the trained model
BEST_MODEL_PATH = f'{os.getcwd()}/runs/segment/fire_and_smoke2/weights/best.pt'
trained_model = YOLO(BEST_MODEL_PATH)

# Select a random image from the dataset's test set
test_image_dir = os.path.join(dataset.location, 'test', 'images')
random_test_image_filename = random.choice(os.listdir(test_image_dir))
TEST_IMAGE_PATH = os.path.join(test_image_dir, random_test_image_filename)
print(f"Using test image: {TEST_IMAGE_PATH}")

# Run inference and directly plot the results
# The 'save=True' option will save the annotated image to a 'runs/segment/predict' directory.
# The 'show=True' option will display the image directly in the notebook.
results = trained_model.predict(
    source=TEST_IMAGE_PATH,
    task='segment',
    conf=0.6,
    iou=0.7,
    save=True,
    show=True,
    name='fire_and_smoke_abstracted_pred'
)

### Inspecting Ground Truth Mask Labels

For YOLOv8 segmentation datasets, the ground truth mask labels are stored in `.txt` files. Each file corresponds to an image and contains the class ID followed by the normalized coordinates of the polygon for each segmented object. Let's inspect a random example.

In [ ]:
# Cell to display a random ground truth label file

import os
import random


# Assuming dataset.location is defined from Cell 3
train_labels_dir = os.path.join(dataset.location, 'train', 'labels')

# Get a list of all label files
label_files = [f for f in os.listdir(train_labels_dir) if f.endswith('.txt')]

# Pick a random label file
random_label_filename = random.choice(label_files)
random_label_filepath = os.path.join(train_labels_dir, random_label_filename)

print(f"Inspecting ground truth label file: {random_label_filepath}")

with open(random_label_filepath, 'r') as f:
    label_content = f.read()

print("\n--- Content of the label file ---")
print(label_content)

In [ ]:
# Load the trained model
BEST_MODEL_PATH = f'{os.getcwd()}/runs/segment/fire_and_smoke2/weights/best.pt'
trained_model = YOLO(BEST_MODEL_PATH)

# Select a random image from the dataset's test set
test_image_dir = os.path.join(dataset.location, 'test', 'images')
random_test_image_filename = random.choice(os.listdir(test_image_dir))
TEST_IMAGE_PATH = os.path.join(test_image_dir, random_test_image_filename)
print(f"Using test image: {TEST_IMAGE_PATH}")

# Run inference and directly plot the results
# The 'save=True' option will save the annotated image to a 'runs/segment/predict' directory.
# The 'show=True' option will display the image directly in the notebook.
# Added show_boxes=False and show_labels=False to display only segmentation masks.
results = trained_model.predict(
    source=TEST_IMAGE_PATH,
    task='segment',
    conf=0.1,
    iou=0.7,
    save=True,
    show=True,
    show_boxes=False,    # Do not display bounding boxes
    show_labels=True,   # Do not display class labels for boxes
    name='fire_and_smoke_abstracted_pred'
)

# Extract the actual output directory from the results object (more robust approach)
# The 'save_dir' attribute of the first result object contains the path to the directory where results were saved.
output_dir = results[0].save_dir

# The filename of the saved image will be the same as the random test image filename
saved_image_path = os.path.join(output_dir, random_test_image_filename)

# Check if the file exists before attempting to display
if os.path.exists(saved_image_path):
    display(Image(filename=saved_image_path, width=600))
else:
    print(f"Error: Saved image not found at {saved_image_path}. Please verify the 'output_dir' matches the actual directory where the results were saved.")
